In [8]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

df = pd.read_csv("../data/processed/population.csv")

census_2012 = {"Gasabo" : 529561, "Musanze": 368267}
census_2022 = {"Gasabo": 879505, "Musanze": 476522}

rows = []
for name in ["Gasabo", "Musanze"]:
    d = df.loc[df["district"] == name].copy()

    wp_2012 = d.loc[d["year"] == 2012, "population"].iloc[0]
    factor = census_2012[name] / wp_2012
    d["calibrated"] = d["population"] * factor

    for version, column in [("original", "population"), ("calibrated", "calibrated")]:
        model = LinearRegression().fit(d[["year"]], np.log(d[column]))
        pred = np.exp(model.predict(pd.DataFrame({"year": [2022]})))[0]
        actual = census_2022[name]
        rows.append({
            "district": name,
            "version": version,
            "factor": round(factor, 4),
            "predicted": round(pred),
            "census_2022": actual,
            "error_%": round((pred - actual) / actual * 100, 1),
    })

pd.DataFrame(rows)

,district,version,factor,predicted,census_2022,error_%
0,Gasabo,original,0.9511,946247,879505,7.6
1,Gasabo,calibrated,0.9511,900014,879505,2.3
2,Musanze,original,0.9875,449337,476522,-5.7
3,Musanze,calibrated,0.9875,443703,476522,-6.9


In [11]:
# Census: 2012 to 2022 ( 10years )
census_rate = (476522 / 368267) ** (1 / 10) - 1

# WorldPop: 2012 to 2020 ( 8years )
wp_rate = (435324 / 372943) ** (1 / 8) -1

print("census growth per year:  ", round(census_rate * 100, 2), "%")
print("WorldPop growth per year:", round(wp_rate * 100, 2), "%")

census growth per year:   2.61 %
WorldPop growth per year: 1.95 %


In [12]:
# Census: 2012 to 2022 ( 10years )
census_rate = (879505 / 529561) ** (1 / 10) - 1

# WorldPop: 2012 to 2020 ( 8years )
wp_rate = (883424 / 556764) ** (1 / 8) -1

print("census growth per year:  ", round(census_rate * 100, 2), "%")
print("WorldPop growth per year:", round(wp_rate * 100, 2), "%")

census growth per year:   5.2 %
WorldPop growth per year: 5.94 %
